# Module 22: Bayesian Statistics in Python

**Tier 3 — Applied Bioinformatics | Module 22**

*Prerequisites: Modules 06 (Statistics), 07 (ML). Familiarity with linear regression.*

---

Bayesian statistical modeling in Python using PyMC, Bambi, and ArviZ. This module covers frequentist vs. Bayesian framing, prior specification, multiple regression, model comparison, mixed-effects models, and GLMs — applied to ecological and biological datasets.

**By the end of this module you will be able to:**
- Contrast frequentist and Bayesian inference
- Specify priors and run prior predictive checks
- Fit Bayesian linear and generalized linear models with PyMC
- Use Bambi for formula-based mixed-effects models
- Compare models with LOO-CV and WAIC via ArviZ
- Fit Poisson and Negative Binomial models for count data

**Attribution:** *Statistical concepts based on Fränzi Korner-Nievergelt's Applied Statistics course (original R implementation). Python implementation by course authors. Uses public palmerpenguins dataset.*

---[< Previous: DNA Copy Number Analysis](../21_Copy_Number_Analysis/21_copy_number_analysis.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: TF Footprinting & Chromatin Accessibility >](../23_TF_Footprinting/23_tf_footprinting.ipynb)---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

try:
    import pymc as pm
    import arviz as az
    import bambi as bmb
    print(f"pymc {pm.__version__}, arviz {az.__version__}, bambi {bmb.__version__}")
except ImportError:
    print("Install: pip install pymc arviz bambi")

# palmerpenguins dataset (public)
try:
    from palmerpenguins import load_penguins
    penguins = load_penguins().dropna()
except ImportError:
    # fallback: simulate equivalent data
    rng = np.random.default_rng(42)
    n = 333
    species = rng.choice(["Adelie","Chinstrap","Gentoo"], n)
    bill_length = rng.normal(43, 5, n)
    bill_depth  = rng.normal(17, 2, n)
    flipper     = rng.normal(200, 14, n)
    body_mass   = 100 * flipper + rng.normal(0, 500, n)
    island = rng.choice(["Biscoe","Dream","Torgersen"], n)
    penguins = pd.DataFrame({"species": species, "bill_length_mm": bill_length,
                              "bill_depth_mm": bill_depth, "flipper_length_mm": flipper,
                              "body_mass_g": body_mass, "island": island, "sex": rng.choice(["male","female"],n)})

print(f"Dataset: {penguins.shape}, species: {penguins['species'].unique()}")
penguins.head()

## 1. Frequentist vs Bayesian Framing

**Frequentist:** Parameters are fixed unknowns. Confidence intervals describe the procedure, not the parameter. p-values test H₀.

**Bayesian:** Parameters are uncertain. We encode prior knowledge as a *prior distribution* p(θ). After observing data y, we update via Bayes' theorem:

$$p(\theta | y) = \frac{p(y | \theta) \cdot p(\theta)}{p(y)}$$

posterior ∝ likelihood × prior

**Key outputs:**
- **Posterior distribution** p(θ|y): full uncertainty over parameter values
- **Credible interval (HDI):** the smallest interval containing X% of the posterior — unlike confidence intervals, we can say "95% probability the parameter is in this range"
- **Posterior predictive:** simulate new data from the model to check fit

In [ ]:
# Example: model penguin body mass from flipper length
df = penguins[["body_mass_g","flipper_length_mm"]].copy()
df = (df - df.mean()) / df.std()  # standardize
df.columns = ["mass_std","flipper_std"]

X = df["flipper_std"].values
y = df["mass_std"].values

# Frequentist OLS
ols = smf.ols("mass_std ~ flipper_std", data=df).fit()
print("=== OLS ===")
print(f"β = {ols.params['flipper_std']:.3f}")
print(f"95% CI: {ols.conf_int().loc['flipper_std'].values.round(3)}")
print(f"p-value: {ols.pvalues['flipper_std']:.4f}")

# Bayesian
print("\n=== Bayesian (PyMC) ===")
try:
    with pm.Model() as linear_model:
        alpha = pm.Normal("alpha", mu=0, sigma=2)
        beta  = pm.Normal("beta",  mu=0, sigma=1)
        sigma = pm.HalfNormal("sigma", sigma=1)
        mu = alpha + beta * X
        obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y)
        idata_lin = pm.sample(1000, tune=500, target_accept=0.9,
                              progressbar=False, random_seed=42)

    summ = az.summary(idata_lin, var_names=["alpha","beta","sigma"], hdi_prob=0.95)
    print(summ[["mean","hdi_2.5%","hdi_97.5%","r_hat"]].round(3))
except Exception as e:
    print(f"PyMC sampling: {e}")
    print("(PyMC requires correct installation; showing structure only)")

## 2. Prior Specification

Choosing priors is the most distinctive Bayesian skill. Priors encode what we know *before* seeing the data.

| Prior type | When to use | Example |
|---|---|---|
| **Weakly informative** | Little domain knowledge; let data dominate | `Normal(0, 1)` on standardized scale |
| **Informative** | Strong prior knowledge (literature, pilot study) | `Normal(0.5, 0.1)` from previous study |
| **Non-informative (flat)** | Avoid — can cause convergence problems | `Uniform(-∞, ∞)` |

**Prior predictive check:** sample from the prior and simulate data. Do the simulated values make sense for your problem? (e.g., negative body masses are impossible)

In [ ]:
try:
    with pm.Model() as prior_check_model:
        alpha = pm.Normal("alpha", mu=0, sigma=5)   # wide prior
        beta  = pm.Normal("beta",  mu=0, sigma=3)   # wide prior
        sigma = pm.HalfNormal("sigma", sigma=2)
        mu = alpha + beta * X
        obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y)
        prior_pred = pm.sample_prior_predictive(200, random_seed=42)

    # Prior predictive samples
    pp_obs = prior_pred.prior_predictive["obs"].values.reshape(-1, len(y))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(pp_obs.flatten(), bins=50, alpha=0.7, color="steelblue")
    axes[0].axvline(y.min(), color="red", lw=2, label=f"Data range")
    axes[0].axvline(y.max(), color="red", lw=2)
    axes[0].set_title("Prior predictive (wide priors)\nbody mass std — does range make sense?")
    axes[0].legend()

    # Now narrow priors
    with pm.Model() as narrow_prior_model:
        alpha = pm.Normal("alpha", mu=0, sigma=1)
        beta  = pm.Normal("beta",  mu=0, sigma=0.5)
        sigma = pm.HalfNormal("sigma", sigma=0.5)
        mu = alpha + beta * X
        obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y)
        narrow_pred = pm.sample_prior_predictive(200, random_seed=42)

    np_obs = narrow_pred.prior_predictive["obs"].values.reshape(-1, len(y))
    axes[1].hist(np_obs.flatten(), bins=50, alpha=0.7, color="steelblue")
    axes[1].axvline(y.min(), color="red", lw=2); axes[1].axvline(y.max(), color="red", lw=2)
    axes[1].set_title("Prior predictive (weakly informative priors)")
    plt.tight_layout(); plt.show()
    print("Weakly informative priors keep predicted values near the data range.")
except Exception as e:
    print(f"Prior predictive check skipped: {e}")

## 3. Multiple Regression and Collinearity

Adding multiple predictors can improve model fit — but collinearity (predictors correlated with each other) inflates uncertainty and makes interpretation difficult.

**Variance Inflation Factor (VIF):** VIF > 5–10 indicates collinearity concern.
$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$
where R²_j is the R² from regressing predictor j on all other predictors.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

multi_df = penguins[["body_mass_g","bill_length_mm","bill_depth_mm","flipper_length_mm"]].dropna()
multi_std = (multi_df - multi_df.mean()) / multi_df.std()

# OLS multiple regression
ols_multi = smf.ols("body_mass_g ~ bill_length_mm + bill_depth_mm + flipper_length_mm",
                     data=penguins).fit()
print(ols_multi.summary().tables[1])

# VIF
X_mat = sm.add_constant(multi_std[["bill_length_mm","bill_depth_mm","flipper_length_mm"]])
vif = pd.DataFrame({
    "feature": X_mat.columns,
    "VIF": [variance_inflation_factor(X_mat.values, i) for i in range(X_mat.shape[1])]
})
print("\nVariance Inflation Factors:")
print(vif.to_string(index=False))
print("\nVIF > 5 suggests collinearity; > 10 is problematic")

## 4. Model Comparison: LOO-CV and WAIC

In Bayesian statistics, we compare models by their **expected log predictive density (ELPD)** — how well each model predicts new data.

| Method | Full name | Notes |
|---|---|---|
| **LOO-CV** | Leave-one-out cross-validation | More robust; prefers fewer parameters |
| **WAIC** | Widely Applicable IC | Faster; sensitive to influential observations |

Both are computed from the posterior using Pareto-smoothed importance sampling (PSIS-LOO).

`az.compare()` returns a table sorted by ELPD — the model with the highest ELPD (least negative) is preferred.

In [ ]:
try:
    # Model 1: flipper length only
    with pm.Model() as m1:
        a = pm.Normal("a", 0, 2); b = pm.Normal("b", 0, 1)
        s = pm.HalfNormal("s", 1)
        pm.Normal("y", mu=a + b*X, sigma=s, observed=y)
        idata_m1 = pm.sample(800, tune=400, progressbar=False, random_seed=1)
        pm.sample_posterior_predictive(idata_m1, extend_inferencedata=True)

    # Model 2: flipper + bill length
    X2 = (penguins["bill_length_mm"] - penguins["bill_length_mm"].mean()) / penguins["bill_length_mm"].std()
    with pm.Model() as m2:
        a  = pm.Normal("a", 0, 2)
        b1 = pm.Normal("b1", 0, 1); b2 = pm.Normal("b2", 0, 1)
        s  = pm.HalfNormal("s", 1)
        pm.Normal("y", mu=a + b1*X + b2*X2.values, sigma=s, observed=y)
        idata_m2 = pm.sample(800, tune=400, progressbar=False, random_seed=2)
        pm.sample_posterior_predictive(idata_m2, extend_inferencedata=True)

    comp = az.compare({"flipper_only": idata_m1, "flipper+bill": idata_m2}, ic="loo")
    print("Model comparison (LOO-CV):")
    print(comp[["elpd_loo","p_loo","d_loo","weight"]].round(2))
    az.plot_compare(comp, insample_dev=False)
    plt.title("LOO-CV model comparison"); plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Model comparison skipped: {e}")
    print("Pattern: az.compare({'m1': idata_m1, 'm2': idata_m2}, ic='loo')")

## 5. Linear Mixed-Effects Models

When data has a grouped structure (measurements nested within individuals, sites, or species), **mixed-effects models** partition variance into fixed effects (population-level) and random effects (group-level).

**Notation (Bambi/lme4):**
- `y ~ x` — fixed effect only
- `y ~ x + (1|group)` — fixed effect + random intercept per group
- `y ~ x + (x|group)` — fixed effect + random slope per group

**Partial pooling:** random effects "share information" across groups — groups with few observations borrow strength from the population estimate.

In [ ]:
try:
    import bambi as bmb

    # Random intercept per species
    m_me = bmb.Model(
        "body_mass_g ~ flipper_length_mm + (1|species)",
        data=penguins,
        family="gaussian"
    )
    idata_me = m_me.fit(draws=800, tune=400, target_accept=0.9,
                         progressbar=False, random_seed=42)

    print("Mixed-effects model summary:")
    summ = az.summary(idata_me, var_names=["flipper_length_mm", "Intercept"], hdi_prob=0.95)
    print(summ[["mean","hdi_2.5%","hdi_97.5%","r_hat"]].round(3))

    # Plot species random intercepts
    az.plot_forest(idata_me, var_names=["1|species"], combined=True)
    plt.title("Random intercepts by species"); plt.tight_layout(); plt.show()

except Exception as e:
    print(f"Mixed-effects skipped: {e}")
    print("Pattern: bmb.Model('y ~ x + (1|group)', data=df).fit(draws=1000)")

## 6. Generalized Linear Models (GLMs)

Not all outcomes are normally distributed. GLMs extend linear models with:
1. A **link function** connecting the linear predictor to the mean
2. A **family** distribution appropriate for the data type

| Family | Link | Use when | R/Python |
|---|---|---|---|
| Gaussian | identity | Continuous, symmetric | `family="gaussian"` |
| Binomial | logit | Binary outcomes (0/1) | `family="bernoulli"` |
| Poisson | log | Count data (no overdispersion) | `pm.Poisson` |
| Negative Binomial | log | Count data with overdispersion | `pm.NegativeBinomial` |
| Zero-Inflated Poisson | log | Excess zeros | `pm.ZeroInflatedPoisson` |

In [ ]:
# Simulate count data: number of nesting attempts
rng = np.random.default_rng(42)
n = 200
temperature = rng.normal(15, 3, n)
# True relationship: log(count) = 1.2 + 0.1*temperature
true_mu = np.exp(1.2 + 0.1 * temperature)
counts_poisson = rng.poisson(true_mu)
# Overdispersed version
alpha_nb = 3.0  # dispersion parameter
counts_negbin = rng.negative_binomial(alpha_nb, alpha_nb / (alpha_nb + true_mu))

print(f"Poisson: mean={counts_poisson.mean():.1f}, var={counts_poisson.var():.1f}")
print(f"NegBin:  mean={counts_negbin.mean():.1f},  var={counts_negbin.var():.1f}")
print(f"Overdispersion ratio: {counts_negbin.var()/counts_negbin.mean():.2f} (Poisson=1.0)")

X_temp = (temperature - temperature.mean()) / temperature.std()

try:
    # Poisson GLM
    with pm.Model() as poisson_glm:
        a = pm.Normal("a", 0, 2); b = pm.Normal("b", 0, 1)
        mu = pm.math.exp(a + b * X_temp)
        pm.Poisson("y", mu=mu, observed=counts_poisson)
        idata_pois = pm.sample(800, tune=400, progressbar=False, random_seed=1)

    # Negative Binomial GLM
    with pm.Model() as negbin_glm:
        a = pm.Normal("a", 0, 2); b = pm.Normal("b", 0, 1)
        alpha = pm.Exponential("alpha", lam=1)
        mu = pm.math.exp(a + b * X_temp)
        pm.NegativeBinomial("y", mu=mu, alpha=alpha, observed=counts_negbin)
        idata_nb = pm.sample(800, tune=400, progressbar=False, random_seed=2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    az.plot_posterior(idata_pois, var_names=["b"], ax=axes[0])
    axes[0].set_title("Poisson GLM: β (temperature effect)")
    az.plot_posterior(idata_nb, var_names=["b"], ax=axes[1])
    axes[1].set_title("Neg. Binomial GLM: β")
    plt.tight_layout(); plt.show()

    print(f"\nTrue β = 0.1 (standardized)")
    print(f"Poisson β estimate: {az.summary(idata_pois, var_names=['b'])['mean']['b']:.3f}")
    print(f"NegBin  β estimate: {az.summary(idata_nb, var_names=['b'])['mean']['b']:.3f}")
except Exception as e:
    print(f"GLM sampling skipped: {e}")

## 7. Advanced: GLMM, Zero-Inflation, GAMs, Meta-analysis

### Generalized Linear Mixed Models (GLMM)
Combine GLM families with random effects:
```python
# Bambi: Poisson GLMM with random intercept
m = bmb.Model("count ~ temperature + (1|site_id)", data=df, family="poisson")
```

### Zero-Inflated Models
When excess zeros occur beyond what Poisson/NegBin predicts:
```python
with pm.Model() as zip_model:
    psi = pm.Beta("psi", 1, 1)  # psi = P(non-zero) i.e. mixture weight on Poisson component
    mu = pm.math.exp(a + b * X)
    pm.ZeroInflatedPoisson("y", psi=psi, mu=mu, observed=y_count)
```

### Generalized Additive Models (GAMs)
Replace linear terms with smooth spline functions:
```python
# pygam (frequentist GAMs)
from pygam import LogisticGAM
gam = LogisticGAM().fit(X, y)
# Bayesian GAM: represent splines as B-spline basis + random effects on coefficients
```

### Bayesian Meta-analysis
Pool effect estimates across studies with partial pooling:
$$\mu_i \sim \text{Normal}(\mu_\text{pop}, \sigma_\text{pop})$$
$$y_i \sim \text{Normal}(\mu_i, \sigma_i)$$
where σᵢ is the known within-study standard error.

In [ ]:
# Simulated meta-analysis: 8 studies measuring the same effect
rng = np.random.default_rng(99)
n_studies = 8
true_effect = 0.4
study_ses   = rng.uniform(0.05, 0.25, n_studies)  # known SEs
study_effects = rng.normal(true_effect, 0.1, n_studies)  # observed estimates

print("Simulated studies:")
df_meta = pd.DataFrame({"study": range(1, n_studies+1),
                          "effect": study_effects.round(3),
                          "se": study_ses.round(3)})
print(df_meta.to_string(index=False))

try:
    with pm.Model() as meta_model:
        mu_pop    = pm.Normal("mu_pop", 0, 1)
        sigma_pop = pm.HalfNormal("sigma_pop", 0.5)
        mu_i = pm.Normal("mu_i", mu_pop, sigma_pop, shape=n_studies)
        pm.Normal("obs", mu=mu_i, sigma=study_ses, observed=study_effects)
        idata_meta = pm.sample(1000, tune=500, progressbar=False, random_seed=7)

    print(f"\nPooled effect estimate: {az.summary(idata_meta, var_names=['mu_pop'])['mean']['mu_pop']:.3f}")
    print(f"True effect: {true_effect}")
    az.plot_posterior(idata_meta, var_names=["mu_pop"])
    plt.title("Posterior: pooled effect (meta-analysis)"); plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Meta-analysis skipped: {e}")

## Summary: Bayesian Modeling Workflow

| Step | Tool | Notes |
|---|---|---|
| Specify model | `pymc` / `bambi` | Formula API or explicit graph |
| Prior predictive | `pm.sample_prior_predictive()` | Check: does prior cover plausible values? |
| Fit | `pm.sample()` or `model.fit()` | Check R-hat < 1.01, ESS > 100 |
| Diagnostics | `az.plot_trace()`, `az.summary()` | Look for divergences, poor mixing |
| Posterior predictive | `pm.sample_posterior_predictive()` | Does model generate data like observed? |
| Model comparison | `az.compare()` | LOO preferred; WAIC as fast alternative |

**Related skill:** `bayesian-python.md`

In [ ]:
try:
    # Show trace plot and pair plot for linear model
    az.plot_trace(idata_lin, var_names=["alpha","beta","sigma"], compact=True)
    plt.suptitle("MCMC trace plots — check for mixing and stationarity")
    plt.tight_layout(); plt.show()

    # R-hat summary
    summ = az.summary(idata_lin, var_names=["alpha","beta","sigma"])
    print("Convergence diagnostics:")
    print(summ[["mean","sd","hdi_2.5%","hdi_97.5%","ess_bulk","r_hat"]].round(3))
    print("\nR-hat close to 1.0 = good convergence; ESS > 100 = sufficient samples")
except Exception as e:
    print(f"Diagnostics plot skipped (requires earlier models): {e}")

In [ ]:
# Exercise 22
# 1. Fit a Bayesian simple linear regression predicting bill_depth_mm from bill_length_mm
#    using the penguins dataset. What does the posterior for β tell you?
#    Is a negative slope plausible? Why might this be (Simpson's paradox)?
# 2. Add species as a random intercept using bambi: "bill_depth_mm ~ bill_length_mm + (1|species)"
#    Compare LOO-CV for the simple vs mixed-effects model.
# 3. Simulate count data with strong overdispersion (var/mean = 5).
#    Fit both Poisson and Negative Binomial GLMs. Compare model fits with LOO.
# 4. Challenge: implement a Bayesian linear model with a horseshoe prior on β
#    (pm.StudentT with ν=1 and σ=σ_local). This is used for sparse regression
#    when many predictors are expected to have zero effect.

---[< Previous: DNA Copy Number Analysis](../21_Copy_Number_Analysis/21_copy_number_analysis.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: TF Footprinting & Chromatin Accessibility >](../23_TF_Footprinting/23_tf_footprinting.ipynb)---